# 06 · Agents and Tool Calling

So far *we* decided the control flow. An **agent** hands that decision to the model: given tools, it decides which to call, with what arguments, and when it's done — the **ReAct** loop (reason → act → observe → repeat).

```
question ─► [LLM reasons] ─► [call tool] ─► [observe result] ─► ... ─► final answer
                 ▲──────────────────────────────────────┘
```

We use LangChain's `create_agent` (the current prebuilt agent in `langchain.agents`, built on LangGraph), which wraps this loop with proper state and observability.

> **Version note:** `create_agent` is the LangChain 1.x replacement for the older
> `langgraph.prebuilt.create_react_agent`. If you're on an older stack, the concepts are identical —
> just the import and the `system_prompt` argument name differ.

---

## Setup

In [1]:
%pip install -qU langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv("../../.env")

# Agents need a tool-calling model; temperature 0 keeps tool choices deterministic.
llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0)

---
## 1. Define tools with `@tool`

The `@tool` decorator turns a plain function into a tool. **The docstring and type hints ARE the schema** the model sees — write them for the model, not just for humans.

In [3]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

@tool
def get_stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker symbol (e.g. 'AAPL')."""
    # A real tool would call an API. We stub a few values.
    prices = {"AAPL": 214.30, "MSFT": 458.10, "GOOG": 178.55}
    if ticker.upper() not in prices:
        raise ValueError(f"Unknown ticker '{ticker}'. Known: {list(prices)}")
    return f"${prices[ticker.upper()]}"

tools = [multiply, word_count, get_stock_price]

### What the model actually sees

Each tool exposes a `name`, `description` (from the docstring), and an `args` schema (from the type hints).

In [4]:
for t in tools:
    print(f"{t.name:<16} args={t.args}\n{'':16} {t.description}\n")

multiply         args={'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
                 Multiply two numbers together.

word_count       args={'text': {'title': 'Text', 'type': 'string'}}
                 Count how many words are in a piece of text.

get_stock_price  args={'ticker': {'title': 'Ticker', 'type': 'string'}}
                 Get the latest price for a stock ticker symbol (e.g. 'AAPL').



---
## 2. Build the agent

`create_agent` wires the LLM + tools into a ready-to-run ReAct loop.

In [5]:
agent = create_agent(llm, tools)

result = agent.invoke({
    "messages": [("human",
        "What is 23.5 times 8? And how many words are in 'the quick brown fox jumps'?")]
})

# The final answer is the last message.
print(result["messages"][-1].content)

The result of 23.5 times 8 is 188.0, and there are 5 words in "the quick brown fox jumps."


---
## 3. See the loop: stream every step

This is where agents earn their keep — you can watch each reason → tool-call → observation.

In [6]:
for step in agent.stream(
    {"messages": [("human", "What's the price of AAPL, and what is that times 10?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the price of AAPL, and what is that times 10?
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (call_vvfbJtxDhfat4qdxeKmvdXOx)
 Call ID: call_vvfbJtxDhfat4qdxeKmvdXOx
  Args:
    ticker: AAPL
  multiply (call_SqSTIGTqa9VRsJWwTqRL15m8)
 Call ID: call_SqSTIGTqa9VRsJWwTqRL15m8
  Args:
    a: 0
    b: 10
================================= Tool Message =================================
Name: multiply

0.0
================================== Ai Message ==================================

The current price of AAPL is $214.3. Multiplying that by 10 gives $2143.


> Notice the trace: the model calls `get_stock_price('AAPL')`, observes `$214.30`, then calls `multiply(214.30, 10)` — it chained two tools on its own. That's the ReAct loop.

---
## 4. Tool error handling

Tools fail — bad input, API errors. When a tool raises, LangGraph feeds the error back to the model as a `ToolMessage` instead of crashing, so the agent can recover or explain.

In [7]:
# 'TSLA' is a typo the tool doesn't know -> it raises -> the agent sees the error and recovers.
res = agent.invoke({"messages": [("human", "What's the price of TSLA?")]})
print(res["messages"][-1].content)

ValueError: Unknown ticker 'TSLA'. Known: ['AAPL', 'MSFT', 'GOOG']

---
## 5. Steer the agent with a system prompt

Pass a `system_prompt` to shape behaviour — persona, constraints, when to use (or avoid) tools.

In [8]:
guided_agent = create_agent(
    llm, tools,
    system_prompt="You are a precise assistant. Always use tools for arithmetic and prices "
                  "instead of guessing. Show the final number clearly.",
)

out = guided_agent.invoke({"messages": [("human", "Price of MSFT times the word count of 'buy low sell high'?")]})
print(out["messages"][-1].content)

The price of MSFT is $458.1, and the word count of "buy low sell high" is 4. Multiplying these gives a total of $1832.4.


---
## What to remember

| Concept | What it does |
|---|---|
| `@tool` | Turns a function into a tool; **docstring + type hints = the schema** |
| `tool.args` / `tool.description` | Exactly what the model reads to decide the call |
| `create_agent(llm, tools)` | Prebuilt ReAct loop (reason → act → observe) with state |
| `stream_mode="values"` | Watch each step of the loop for observability/debugging |
| Errors → `ToolMessage` | A raising tool is fed back to the model, not crashed |
| `system_prompt=...` | System instructions that steer tool use and persona |

**vs. the earlier patterns:** we no longer choose the control flow — the *model* decides which tool to call and when to stop. That flexibility is powerful but less predictable, so keep tools well-described and cap the work.

**Next (Phase: State & memory):** 07 — **Checkpointer vs. store**: give your agent short-term and long-term memory.